# Task SpellingBee

In [12]:
import os
import re
import json
import random
import requests

In [2]:
def print_colored(convo, limit=float('inf')):
    for i, message in enumerate(convo['messages']):
        if i >= limit:
            print(f"\033[31m... {len(convo['messages']) - limit} more messages ...\033[0m")
            break
        role = message['role']
        content = message['content']
        if role == 'system':
            print(f"\033[33m{content}\033[0m")  # yellow
        elif role == 'assistant':
            if isinstance(content, str):
                print(f"\033[34m{content}\033[0m")  # blue
            elif isinstance(content, list):
                for part in content:
                    if part['type'] == 'text':
                        print(f"\033[34m{part['text']}\033[0m", end='')  # blue
                    elif part['type'] == 'python':
                        print(f"\033[36m{part['text']}\033[0m", end='')  # cyan
                    elif part['type'] == 'python_output':
                        print(f"\033[35m{part['text']}\033[0m", end='')  # magenta
                    else:
                        print(f"\033[31m{part}\033[0m", end='')  # red for unknown part types
                print()  # newline after the assistant message
            else:
                # red
                print(f"\033[31m{content}\033[0m")  # red
                
        elif role == 'user':
            print(f"\033[32m{content}\033[0m")  # green
        else:
            print(f"\033[31m{content}\033[0m")  # red

In [3]:
filepath = "words_alpha.txt"
if not os.path.exists(filepath):
    # 370K English words
    url = "https://raw.githubusercontent.com/dwyl/english-words/refs/heads/master/words_alpha.txt"
    r = requests.get(url)
    r.raise_for_status()
    with open(filepath, "w") as f:
        f.write(r.text)
with open(filepath, "r") as f:
    words = [line.strip() for line in f.readlines() if line.strip()]
print(f"Loaded {len(words)} words")

Loaded 370105 words


In [4]:
for _ in range(10):
    word = random.choice(words)
    print(f"Word: {word}")

Word: meatless
Word: begrudges
Word: uncountermanded
Word: uncreate
Word: patavinity
Word: moonal
Word: custodee
Word: unsocialising
Word: ladyfern
Word: inogen


In [5]:
# User message templates - adopted from Nanochat's spellingbee.py
USER_MSG_TEMPLATES = [
    "How many {letter} are in the word {word}",
    "How many {letter} are in {word}",
    "Count the number of {letter} in {word}",
    "How many times does {letter} appear in {word}",
    "What's the count of {letter} in {word}",
    "In the word {word}, how many {letter} are there",
    "How many letter {letter} are in the word {word}",
    "Count how many {letter} appear in {word}",
    "Tell me the number of {letter} in {word}",
    "How many occurrences of {letter} are in {word}",
    "Find the count of {letter} in {word}",
    "Can you count the {letter} letters in {word}",
    "What is the frequency of {letter} in {word}",
    "How many {letter}s are in {word}",
    "How many {letter}'s are in {word}",
    "Count all the {letter} in {word}",
    "How many times is {letter} in {word}",
    "Number of {letter} in {word}",
    "Total count of {letter} in {word}",
    "How many {letter} does {word} have",
    "How many {letter} does {word} contain",
    "What's the number of {letter} in {word}",
    "{word} has how many {letter}",
    "In {word}, count the {letter}",
    "How many {letter} appear in {word}",
    "Count the {letter} in {word}",
    "Give me the count of {letter} in {word}",
    "How many instances of {letter} in {word}",
    "Show me how many {letter} are in {word}",
    "Calculate the number of {letter} in {word}",
    # Spanish
    "¿Cuántas {letter} hay en {word}?",
    "¿Cuántas veces aparece {letter} en {word}?",
    "Cuenta las {letter} en {word}",
    "¿Cuántas letras {letter} tiene {word}?",
    # Chinese (Simplified)
    "{word}中有多少个{letter}",
    "{word}里有几个{letter}",
    "数一下{word}中的{letter}",
    "{word}这个词里有多少{letter}",
    # Korean
    "{word}에 {letter}가 몇 개 있나요",
    "{word}에서 {letter}의 개수는",
    "{word}에 {letter}가 몇 번 나오나요",
    "{word}라는 단어에 {letter}가 몇 개",
    # French
    "Combien de {letter} dans {word}",
    "Combien de fois {letter} apparaît dans {word}",
    "Compte les {letter} dans {word}",
    # German
    "Wie viele {letter} sind in {word}",
    "Wie oft kommt {letter} in {word} vor",
    "Zähle die {letter} in {word}",
    # Japanese
    "{word}に{letter}は何個ありますか",
    "{word}の中に{letter}がいくつ",
    "{word}に{letter}が何回出てくる",
]

In [8]:
seed = 0

test_random_seed_offset = 10_000_000
rng = random.Random(seed)
# pick random letter from word (90% prob) or random letter overall (10% prob)
letter = rng.choice(word) if rng.random() < 0.9 else rng.choice("abcdefghijklmnopqrstuvwxyz")
real_count = word.count(letter)
print(f"Word: {word}, Letter: {letter}, Real count: {real_count}")

Word: inogen, Letter: g, Real count: 1


In [9]:
# Create User Message
template = rng.choice(USER_MSG_TEMPLATES)
if rng.random() < 0.3:    # 30% chance lowercase (not everyone capitalizes)
    template = template.lower()
quotes = ['', "'", '"']
letter_quote = rng.choice(quotes)
word_quote = rng.choice(quotes)
letter_maybe_quoted = f"{letter_quote}{letter}{letter_quote}"
word_maybe_quoted = f"{word_quote}{word}{word_quote}"
user_msg = template.format(letter=letter_maybe_quoted, word=word_maybe_quoted)
if rng.random() < 0.5:    # 50% question mark (not everyone uses question marks)
    user_msg += "?"
print(f"User message: {user_msg}")

User message: count the number of "g" in 'inogen'?


In [ ]:
# Hide inside class for general cleanliness
# https://github.com/openai/grade-school-math/blob/3101c7d5072418e28b9008a6636bde82a006892c/grade_school_math/dataset.py#L28
GSM_ANS_RE = re.compile(r"#### (\-?[0-9\.\,]+)")
def extract_answer(assistant_response):
    match = GSM_ANS_RE.search(assistant_response)
    if match:
        match_str = match.group(1).strip()
        match_str = match_str.replace(",", "")
        return match_str
    return None

In [14]:
# Create Assistant Message
word_letters = ",".join(list(word))
manual_text = \
f"""We are asked to find the number '{letter}' in the word '{word}'. Let me try a manual approach first.

First spell the word out:
{word}:{word_letters}

Then count the occurrences of '{letter}':
"""
running_count = 0
for i, c in enumerate(word):
    if c == letter:
        running_count += 1
        # no space before char, ' a', 'a' are different tokens
        manual_text += f"{i}:{c} hit! count={running_count}\n"
    else:
        manual_text += f"{i}:{c}\n"

manual_text += f"\nThis gives us {running_count}."
# Part 1: Manual counting
assistant_parts = []
assistant_parts.append({"type": "text", "text": manual_text})
# Part 2: Python verification text
assistant_parts.append({"type": "text", "text": "\n\nLet me double check this using Python:\n\n"})
# Part 3: Python tool call
python_expr = f"'{word}'.count('{letter}')"
assistant_parts.append({"type": "python", "text": python_expr})
# Part 4: Python tool output
assistant_parts.append({"type": "python_output", "text": str(real_count)})
# Part 5: Final answer
assistant_parts.append({"type": "text", "text": f"\n\nPython gives us {real_count}.\n\nMy final answer is:\n\n#### {real_count}"})

# Extract answer
last_part = assistant_parts[-1]
expected_answer = extract_answer(last_part['text'])
int(expected_answer)  # throws if not an integer

messages = [
    {"role": "user", "content": user_msg},
    {"role": "assistant", "content": assistant_parts}
]
convo = {
    "messages": messages,
    "eval": {
        "answer": expected_answer
    }
}
print_colored(convo, limit=10)

count the number of "g" in 'inogen'?
We are asked to find the number 'g' in the word 'inogen'. Let me try a manual approach first.

First spell the word out:
inogen:i,n,o,g,e,n

Then count the occurrences of 'g':
0:i
1:n
2:o
3:g hit! count=1
4:e
5:n

This gives us 1.

Let me double check this using Python:

'inogen'.count('g')1

Python gives us 1.

My final answer is:

#### 1


In [16]:
class TaskSpellingBee:
    def __init__(self, filepath, split, stop=None):
        assert split in ["train", "test"]
        with open(filepath, "r") as f:
            self.words = [line.strip() for line in f.readlines() if line.strip()]
        self.split = split
        self.length = stop if stop is not None else len(self.words)

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        if idx >= self.length:
            raise IndexError(idx)
        # Weird way to split train/valid inherited from Nanochat (which may have inherited it from SpellingBee)
        # I'm keeping in like this for now to keep equivalence with Nanochat
        test_random_seed_offset = 10_000_000
        seed = idx if self.split == "train" else idx + test_random_seed_offset
        rng = random.Random(seed)
        word = rng.choice(self.words)
        # pick random letter from word (90% prob) or random letter overall (10% prob)
        letter = rng.choice(word) if rng.random() < 0.9 else rng.choice("abcdefghijklmnopqrstuvwxyz")
        real_count = word.count(letter)

        # Create User Message
        template = rng.choice(USER_MSG_TEMPLATES)
        if rng.random() < 0.3:    # 30% chance lowercase (not everyone capitalizes)
            template = template.lower()
        quotes = ['', "'", '"']
        letter_quote = rng.choice(quotes)
        word_quote = rng.choice(quotes)
        letter_maybe_quoted = f"{letter_quote}{letter}{letter_quote}"
        word_maybe_quoted = f"{word_quote}{word}{word_quote}"
        user_msg = template.format(letter=letter_maybe_quoted, word=word_maybe_quoted)
        if rng.random() < 0.5:    # 50% question mark (not everyone uses question marks)
            user_msg += "?"

        # Create Assistant Message
        word_letters = ",".join(list(word))
        manual_text = \
f"""We are asked to find the number '{letter}' in the word '{word}'. Let me try a manual approach first.

First spell the word out:
{word}:{word_letters}

Then count the occurrences of '{letter}':
"""
        running_count = 0
        for i, c in enumerate(word, 1):  # count starting from 1
            if c == letter:
                running_count += 1
                # no space before char, ' a', 'a' are different tokens
                manual_text += f"{i}:{c} hit! count={running_count}\n"
            else:
                manual_text += f"{i}:{c}\n"

        manual_text += f"\nThis gives us {running_count}."
        # Part 1: Manual counting
        assistant_parts = []
        assistant_parts.append({"type": "text", "text": manual_text})
        # Part 2: Python verification text
        assistant_parts.append({"type": "text", "text": "\n\nLet me double check this using Python:\n\n"})
        # Part 3: Python tool call
        python_expr = f"'{word}'.count('{letter}')"
        assistant_parts.append({"type": "python", "text": python_expr})
        # Part 4: Python tool output
        assistant_parts.append({"type": "python_output", "text": str(real_count)})
        # Part 5: Final answer
        assistant_parts.append({"type": "text", "text": f"\n\nPython gives us {real_count}.\n\nMy final answer is:\n\n#### {real_count}"})

        # Extract answer
        last_part = assistant_parts[-1]
        expected_answer = extract_answer(last_part['text'])
        int(expected_answer)  # throws if not an integer

        messages = [
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": assistant_parts}
        ]
        result = {
            "messages": messages,
            "eval": {
                "answer": expected_answer
            }
        }
        return result

    def evaluate(self, assistant_response, eval_data):
        assert isinstance(assistant_response, str)
        extracted_answer = extract_answer(assistant_response)
        return extracted_answer == eval_data["answer"]

In [17]:
def check_schema(convo):
    assert isinstance(convo, dict)
    assert convo.keys() == {'messages', 'eval'}
    assert isinstance(convo['messages'], list)
    for message in convo['messages']:
        assert isinstance(message, dict)
        assert message.keys() == {'role', 'content'}
        assert message['role'] in {'system', 'assistant', 'user'}
        if isinstance(message['content'], str):
            assert isinstance(message['content'], str)
        elif isinstance(message['content'], list):
            for part in message['content']:
                assert isinstance(part, dict)
                assert part.keys() == {'type', 'text'}
                assert part['type'] in {'text', 'python', 'python_output'}
                assert isinstance(part['text'], str)
        else:
            assert False, f"Invalid content type: {type(message['content'])}"
        assert len(message['content']) > 0
    assert isinstance(convo['eval'], dict)
    assert 'answer' in convo['eval']

In [18]:
task = TaskSpellingBee("words_alpha.txt", "train")
for i, e in enumerate(task):
    check_schema(e)
    assistant_response = e['messages'][-1]['content'][-1]['text']  # last part of assistant response
    assert task.evaluate(assistant_response, e['eval'])
    assert not task.evaluate("X", e['eval'])  # wrong answer
    if i % 10_000 == 0:
        print(f"Checked {i} / {len(task)} examples...")

Checked 0 / 370105 examples...
Checked 10000 / 370105 examples...
Checked 20000 / 370105 examples...
Checked 30000 / 370105 examples...
Checked 40000 / 370105 examples...
Checked 50000 / 370105 examples...
Checked 60000 / 370105 examples...
Checked 70000 / 370105 examples...
Checked 80000 / 370105 examples...
Checked 90000 / 370105 examples...
Checked 100000 / 370105 examples...
Checked 110000 / 370105 examples...
Checked 120000 / 370105 examples...
Checked 130000 / 370105 examples...
Checked 140000 / 370105 examples...
Checked 150000 / 370105 examples...
Checked 160000 / 370105 examples...
Checked 170000 / 370105 examples...
Checked 180000 / 370105 examples...
Checked 190000 / 370105 examples...
Checked 200000 / 370105 examples...
Checked 210000 / 370105 examples...
Checked 220000 / 370105 examples...
Checked 230000 / 370105 examples...
Checked 240000 / 370105 examples...
Checked 250000 / 370105 examples...
Checked 260000 / 370105 examples...
Checked 270000 / 370105 examples...
Checke

In [19]:
task = TaskSpellingBee("words_alpha.txt", "test")
for i, e in enumerate(task):
    check_schema(e)
    assistant_response = e['messages'][-1]['content'][-1]['text']  # last part of assistant response
    assert task.evaluate(assistant_response, e['eval'])
    assert not task.evaluate("X", e['eval'])  # wrong answer
    if i % 10_000 == 0:
        print(f"Checked {i} / {len(task)} examples...")

Checked 0 / 370105 examples...
Checked 10000 / 370105 examples...
Checked 20000 / 370105 examples...
Checked 30000 / 370105 examples...
Checked 40000 / 370105 examples...
Checked 50000 / 370105 examples...
Checked 60000 / 370105 examples...
Checked 70000 / 370105 examples...
Checked 80000 / 370105 examples...
Checked 90000 / 370105 examples...
Checked 100000 / 370105 examples...
Checked 110000 / 370105 examples...
Checked 120000 / 370105 examples...
Checked 130000 / 370105 examples...
Checked 140000 / 370105 examples...
Checked 150000 / 370105 examples...
Checked 160000 / 370105 examples...
Checked 170000 / 370105 examples...
Checked 180000 / 370105 examples...
Checked 190000 / 370105 examples...
Checked 200000 / 370105 examples...
Checked 210000 / 370105 examples...
Checked 220000 / 370105 examples...
Checked 230000 / 370105 examples...
Checked 240000 / 370105 examples...
Checked 250000 / 370105 examples...
Checked 260000 / 370105 examples...
Checked 270000 / 370105 examples...
Checke